In [ ]:
!pip install -U datasets

In [ ]:
!nvidia-smi

In [ ]:
import os
import requests
import sentencepiece as spm

model_path = 'e8_morpheme.model'   # имя файла оставлено для совместимости

if not os.path.exists(model_path):
    print("🔧 Токенайзер не найден. Обучаем на Shakespeare...")
    url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
    text = requests.get(url).text
    temp_file = "input_text.txt"
    with open(temp_file, "w", encoding="utf-8") as f:
        f.write(text)
    spm.SentencePieceTrainer.train(
        input=temp_file,
        model_prefix='e8_morpheme',
        vocab_size=2048,
        model_type='bpe',
        character_coverage=1.0,
        byte_fallback=True,
        unk_id=0, pad_id=1, bos_id=2, eos_id=3
    )
    print("✅ Токенайзер обучен и сохранён как e8_morpheme.model")
else:
    print("✅ Токенайзер уже существует, загружаем...")

sp = spm.SentencePieceProcessor(model_file='e8_morpheme.model')
vocab_size = sp.get_piece_size()
print(f"💎 Vocab Size: {vocab_size}")

In [ ]:
import gc
import torch
import torch.nn as nn
import torch.nn.functional as F
import requests
import os
import re
import math
from dataclasses import dataclass
from typing import Optional
import time
from torch import Tensor
import sentencepiece as spm
import io
from datasets import load_dataset
import random
from collections import deque
import numpy as np

# 0. АВТО-ОПРЕДЕЛЕНИЕ GPU
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"📡 Device: {device} | {'🔥 GPU ACTIVE' if device == 'cuda' else '⚠️ CPU MODE'}")

# 1. ФУНКЦИИ КОДИРОВАНИЯ/ДЕКОДИРОВАНИЯ
def encode(text):
    return sp.encode(text)

def decode(tokens):
    return sp.decode(tokens)

# 2. ПОДКЛЮЧЕНИЕ К БЕСКОНЕЧНОМУ ПОТОКУ TINYSTORIES
#dataset = load_dataset("roneneldan/TinyStories", streaming=True, split="train")

dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
train_iter = iter(dataset)
print("🌊 Стриминг fineweb-edu активирован")

# 3. ФУНКЦИЯ ПОЛУЧЕНИЯ БАТЧА ИЗ ПОТОКА (с буфером для перемешивания)
def get_batch_streaming(iterator, batch_size, block_size, device, pad_token_id=1, buffer_size=200):
    x_batch, y_batch = [], []
    buffer = deque()

    while len(buffer) < buffer_size:
        try:
            ex = next(iterator)
            buffer.append(ex)
        except StopIteration:
            break

    while len(x_batch) < batch_size:
        if not buffer:
            return None, None
        ex = random.choice(buffer)
        tokens = encode(ex['text'])
        if len(tokens) <= 1:
            continue

        if len(tokens) > block_size + 1:
            start = random.randint(0, len(tokens) - block_size - 1)
            chunk = tokens[start:start + block_size + 1]
        else:
            pad_len = block_size + 1 - len(tokens)
            chunk = tokens + [pad_token_id] * pad_len

        x_batch.append(chunk[:-1])
        y_batch.append(chunk[1:])

    try:
        new_ex = next(iterator)
        buffer.append(new_ex)
        buffer.popleft()
    except StopIteration:
        pass

    X = torch.tensor(x_batch, dtype=torch.long, device=device)
    Y = torch.tensor(y_batch, dtype=torch.long, device=device)
    return X, Y

# Пробный батч
xb, yb = get_batch_streaming(train_iter, batch_size=4, block_size=256, device=device)
print(f"✅ Пробный батч: X {xb.shape}, Y {yb.shape} | токенов: {xb.numel()}")

In [ ]:
from torch.optim.lr_scheduler import LinearLR, CosineAnnealingLR, SequentialLR
# ==================== КОНФИГУРАЦИЯ LEECH ====================
@dataclass
class LeechConfig:
    vocab_size: int
    d_model: int = 384    # 384 / 8 = 48 (кратно 24) -> head_dim = 48
    n_layers: int = 12
    n_heads: int = 8
    block_size: int = 512
    dropout: float = 0.05
    bias: bool = False
    tie_weights: bool = True
    lambda_geo: float = 0.01    # вес геометрической потери (0 = отключена)
    resonance_threshold: float = 0.95   # порог для детекции «сна»

    def __post_init__(self):
        assert self.d_model % self.n_heads == 0
        head_dim = self.d_model // self.n_heads
        assert head_dim % 24 == 0, "head_dim должен быть кратен 24"
# ==================== Leech kernel ====================
def generate_leech_kernel(dim=24):
    """Генерирует ортогональную матрицу 24x24 (ядро Лича)."""
    base = np.zeros((dim, dim))
    for i in range(dim - 1):
        base[i, i], base[i, i+1] = 2, 2
    base[-1, -1], base[-1, 0] = 2, -2
    q, _ = np.linalg.qr(base)
    return torch.from_numpy(q).float()

# ==================== ВНИМАНИЕ С ЯДРОМ ЛИЧА ====================
class LeechAttention(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.n_heads = cfg.n_heads
        self.head_dim = cfg.d_model // cfg.n_heads
        self.scale = self.head_dim ** -0.5
        self.num_blocks = self.head_dim // 24 # число 24‑мерных блоков в одной голове

        kernel = generate_leech_kernel(24)  # [24, 24]
        total_blocks = self.n_heads * self.num_blocks
        W_list = [kernel] * total_blocks
        self.register_buffer('W_leech', torch.block_diag(*W_list))  # блочно-диагональная

        self.qkv = nn.Linear(cfg.d_model, 3 * cfg.d_model, bias=cfg.bias)
        self.out = nn.Linear(cfg.d_model, cfg.d_model, bias=cfg.bias)
        self.dropout = nn.Dropout(cfg.dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(1, 1, cfg.block_size, cfg.block_size)))

    def forward(self, x):
        B, T, _ = x.shape
        qkv = self.qkv(x).reshape(B, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)
        q, k, v = qkv.unbind(0)   # [B, n_heads, T, head_dim]

        # Разбиваем head_dim на блоки по 24 и применяем ядро
        q = q.view(B, self.n_heads, T, self.num_blocks, 24)
        k = k.view(B, self.n_heads, T, self.num_blocks, 24)
        kernel = self.W_leech[0:24, 0:24]   # [24,24] (одинаково для всех блоков)
        q = torch.einsum('...i,ij->...j', q, kernel)
        k = torch.einsum('...i,ij->...j', k, kernel)
        q = q.reshape(B, self.n_heads, T, self.head_dim)
        k = k.reshape(B, self.n_heads, T, self.head_dim)

        scores = (q @ k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(self.causal_mask[:,:,:T,:T] == 0, float('-inf'))
        attn = F.softmax(scores, dim=-1)
        attn = self.dropout(attn)
        out = (attn @ v).transpose(1, 2).reshape(B, T, -1)
        return self.out(out)

# ==================== БЛОК ТРАНСФОРМЕРА ====================
class LeechBlock(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.ln1 = nn.LayerNorm(cfg.d_model)
        self.attn = LeechAttention(cfg)
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = nn.Sequential(
            nn.Linear(cfg.d_model, 4 * cfg.d_model),
            nn.GELU(),
            nn.Linear(4 * cfg.d_model, cfg.d_model),
            nn.Dropout(cfg.dropout)
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class LeechResonanceBiasing(nn.Module):
    """
    Добавляет к логитам смещение, пропорциональное резонансу скрытых состояний с базисом Лича.
    Работает как для одного вектора, так и для последовательности (B, T, d_model).

    LeechResonanceLoss – для основного обучения (без активного резонатора). Он добавляет геометрическую регуляризацию к скрытым состояниям.

    LeechResonanceBiasing – для дообучения или генерации с активным резонансом. Он модифицирует логиты напрямую.

    В forward модели мы добавили условие: геометрическая потеря применяется только если use_resonator=False. Это предотвращает двойной учёт геометрии.

    """
    def __init__(self, d_model, vocab_size, leech_basis, alpha_init=0.1):
        super().__init__()
        self.d_model = d_model
        self.vocab_size = vocab_size
        self.num_blocks = d_model // 24
        self.register_buffer('basis', leech_basis)          # [24, 24]
        self.alpha = nn.Parameter(torch.tensor(alpha_init)) # обучаемый коэффициент
        self.register_buffer('token_proj', torch.zeros(vocab_size, 24))

    @torch.no_grad()
    def compute_token_proj(self, tok_emb):
        """
        Вычисляет проекцию эмбеддингов токенов на базис Лича.
        tok_emb: [vocab_size, d_model]
        """
        emb_blocks = tok_emb.view(-1, self.num_blocks, 24)   # [V, K, 24]
        proj = torch.einsum('vki,ij->vkj', emb_blocks, self.basis)  # [V, K, 24]
        proj = proj.sum(dim=1)                                # [V, 24]
        self.token_proj.copy_(proj)

    def forward(self, hidden_states):
        """
        hidden_states: [B, T, d_model] или [B, d_model]
        return: [B, T, vocab_size] или [B, vocab_size] смещение для логитов
        """
        if hidden_states.dim() == 2:
            hidden_states = hidden_states.unsqueeze(1)       # [B, 1, d_model]
        B, T, D = hidden_states.shape
        h_blocks = hidden_states.view(B, T, self.num_blocks, 24)  # [B, T, K, 24]
        h_proj = torch.einsum('btki,ij->btkj', h_blocks, self.basis)  # [B, T, K, 24]
        h_proj = h_proj.sum(dim=2)                            # [B, T, 24]
        bias = self.alpha * (h_proj @ self.token_proj.T)      # [B, T, vocab_size]
        if T == 1:
            bias = bias.squeeze(1)                            # [B, vocab_size]
        return bias

# ==================== ГЕОМЕТРИЧЕСКАЯ ПОТЕРЯ РЕЗОНАНСА ====================
class LeechResonanceLoss(nn.Module):
    def __init__(self, leech_basis):
        super().__init__()
        self.register_buffer('basis', leech_basis)   # [24,24]

    def forward(self, hidden_states):
        """
        hidden_states: [B, T, d_model]
        возвращает скаляр: 1 - среднее макс. косинусное сходство с базисом
        """
        B, T, D = hidden_states.shape
        h = hidden_states.view(B, T, D // 24, 24)          # [B, T, K, 24]
        h_norm = F.normalize(h, dim=-1)
        b_norm = F.normalize(self.basis, dim=-1)           # [24,24]
        sim = torch.matmul(h_norm, b_norm.T)                # [B,T,K,24]
        max_sim = torch.max(sim, dim=-1)[0]                 # [B,T,K]
        return 1.0 - max_sim.mean()

# ==================== ДЕКОДЕР СНОВ (МОНИТОРИНГ) ====================

class DreamDecoder:
    def __init__(self, leech_basis, threshold=0.95):
        self.basis = leech_basis
        self.threshold = threshold

    def check(self, hidden_state):
        """
        hidden_state: [d_model] – последнее скрытое состояние.
        Возвращает (резонанс, статус).
        """
        h = hidden_state[:24].unsqueeze(0)          # [1,24] для простоты берём первые 24
        h_norm = F.normalize(h, dim=-1)
        b_norm = F.normalize(self.basis, dim=-1)
        sim = torch.matmul(h_norm, b_norm.T)
        max_res = torch.max(sim).item()
        if max_res > 0.999:
            status = "ABSOLUTE GENESIS"
        elif max_res > self.threshold:
            status = "AWAKE"
        else:
            status = "DREAMING"
        return max_res, status

# ==================== ПОЛНАЯ МОДЕЛЬ LeechGPT ====================
class LeechGPT(nn.Module):
    def __init__(self, cfg: LeechConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.pos_emb = nn.Parameter(torch.zeros(1, cfg.block_size, cfg.d_model))
        self.blocks = nn.ModuleList([LeechBlock(cfg) for _ in range(cfg.n_layers)])
        self.final_norm = nn.LayerNorm(cfg.d_model)
        self.head = nn.Linear(cfg.d_model, cfg.vocab_size, bias=False)

        # Ядро Лича для потери и мониторинга
        leech_basis = generate_leech_kernel(24)
        self.register_buffer('leech_basis', leech_basis)
        self.resonance_loss_fn = LeechResonanceLoss(leech_basis)

        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.trunc_normal_(module.weight, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.trunc_normal_(module.weight, std=0.02)

    def forward(self, idx, targets=None, use_resonator=False):
        b, t = idx.size()
        assert t <= self.cfg.block_size
        x = self.tok_emb(idx) + self.pos_emb[:, :t, :]
        for block in self.blocks:
            x = block(x)
        x = self.final_norm(x)
        logits = self.head(x)
        hidden = x

        if use_resonator and hasattr(self, 'resonator') and self.resonator is not None:
            bias = self.resonator(hidden)
            logits = logits + bias

        loss = None
        if targets is not None:
            # Сначала вычисляем кросс-энтропию
            loss = F.cross_entropy(logits.view(-1, self.cfg.vocab_size), targets.view(-1))
            # Затем добавляем геометрическую потерю, если нужно
            if self.cfg.lambda_geo > 0 and not use_resonator:
                loss_geo = self.resonance_loss_fn(hidden)
                loss = loss + self.cfg.lambda_geo * loss_geo

        return logits, hidden, loss



    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None, repetition_penalty=1.5, repetition_window=50):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.cfg.block_size else idx[:, -self.cfg.block_size:]
            logits, _, _ = self(idx_cond)          # <-- исправлено
            logits = logits[:, -1, :] / temperature

            if repetition_penalty != 1.0:
                past_tokens = idx[0, -repetition_window:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits[0, t] -= repetition_penalty * count

            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

# ==================== ФУНКЦИЯ ГЕНЕРАЦИИ С МОНИТОРИНГОМ ====================
def leech_generate(model, start_tokens, max_len=100, temperature=0.8,
                   resonance_check=True, leech_basis=None, threshold=0.95):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([start_tokens], device=device)
    if resonance_check:
        decoder = DreamDecoder(leech_basis, threshold)

    print("--- LEECH GENERATION ---")
    with torch.no_grad():
        for step in range(max_len):
            logits, hidden, _ = model(input_ids)          # получаем hidden
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)

            if resonance_check:
                last_hidden = hidden[0, -1, :]            # последнее скрытое состояние
                res, status = decoder.check(last_hidden)
                print(f"Step {step:02d} | Resonance: {res:.6f} | Status: {status}")
    return input_ids

  # ==================== СОЗДАНИЕ МОДЕЛИ ====================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
cfg = LeechConfig(vocab_size=vocab_size)
model = LeechGPT(cfg).to(device)

print(f"💎 Модель Leech-GPT создана.")
print(f"📦 Параметров: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
print(f"   Архитектура: d_model={cfg.d_model}, layers={cfg.n_layers}, heads={cfg.n_heads}")


In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os
import csv
from pathlib import Path

def compute_stable_rank(weight_matrix):
    """
    Вычисляет stable rank (эффективный ранг) матрицы.
    stable_rank = (sum(s_i^2)) / (s_0^2), где s_i - сингулярные числа.
    """
    if len(weight_matrix.shape) > 2:
        weight_matrix = weight_matrix.view(weight_matrix.shape[0], -1)

    # Используем torch.linalg.svd для получения сингулярных чисел
    U, S, V = torch.linalg.svd(weight_matrix, full_matrices=False)

    # Stable rank = сумма квадратов / квадрат первого сингулярного числа
    stable_rank = (S**2).sum() / (S[0]**2)
    condition_number = S[0] / S[-1]

    return stable_rank.item(), condition_number.item(), S.detach().cpu().numpy()

def log_stable_rank(model, step, log_dir, layer_names=None, save_plot=False):
    """
    Логирует stable rank для заданных слоёв модели.

    Args:
        model: модель
        step: текущий шаг обучения
        log_dir: директория для сохранения логов
        layer_names: список имён слоёв для анализа (если None, используется дефолтный)
        save_plot: сохранять ли график спектра
    """
    if layer_names is None:
        layer_names = ['blocks.0.attn.qkv.weight',
                       'blocks.5.attn.qkv.weight',
                       'blocks.11.attn.qkv.weight']

    log_path = Path(log_dir) / 'stable_rank_log.csv'
    plot_dir = Path(log_dir) / 'spectrum_plots'

    # Создаём директории, если их нет
    os.makedirs(log_dir, exist_ok=True)
    if save_plot:
        os.makedirs(plot_dir, exist_ok=True)

    # Заголовки для CSV (если файл не существует)
    file_exists = os.path.isfile(log_path)

    results = {'step': step}

    for layer_name in layer_names:
        if layer_name not in model.state_dict():
            print(f"Предупреждение: слой {layer_name} не найден, пропускаем")
            continue

        weights = model.state_dict()[layer_name].detach().cpu().float()
        sr, cn, spectrum = compute_stable_rank(weights)

        results[f'{layer_name}_sr'] = sr
        results[f'{layer_name}_cn'] = cn

        # Опционально сохраняем график спектра
        if save_plot:
            plt.figure(figsize=(10, 5))

            plt.subplot(1, 2, 1)
            plt.plot(spectrum, 'b-o', markersize=2)
            plt.title(f'SVD Spectrum: {layer_name} at step {step}')
            plt.ylabel('Singular Value')
            plt.grid(True, alpha=0.3)

            plt.subplot(1, 2, 2)
            plt.semilogy(spectrum, 'r-o', markersize=2)
            plt.title('Log Scale')
            plt.ylabel('Log Magnitude')
            plt.grid(True, alpha=0.3)

            plt.tight_layout()
            plt.savefig(plot_dir / f'{layer_name.replace(".", "_")}_step_{step}.png', dpi=100)
            plt.close()

    # Запись в CSV
    with open(log_path, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results.keys())
        if not file_exists:
            writer.writeheader()
        writer.writerow(results)

    print(f"[Step {step}] Stable rank logged for {len(results)-1} layers")
    return results

# Пример интеграции в тренировочный цикл
def train_with_sr_logging(model, train_loader, optimizer, scheduler, total_steps,
                          log_dir, sr_log_freq=5000):
    """
    Пример того, как можно интегрировать логирование в твой цикл.
    """
    for step in range(total_steps):
        # ... твой код обучения ...

        # Логируем stable rank каждые sr_log_freq шагов
        if step % sr_log_freq == 0:
            log_stable_rank(model, step, log_dir, save_plot=True)

        # ... остальной код ...

# Если ты хочешь просто добавить вызов в существующий цикл,
# вставь эту строку в нужном месте:
# if step % 5000 == 0:
#     log_stable_rank(model, step, '/content/drive/MyDrive/leech_tinystories_checkpoints/sr_logs')

In [ ]:
# ======================== DRIVE И ЧЕКПОИНТЫ ========================
from google.colab import drive
drive.mount('/content/drive')

checkpoint_dir = '/content/drive/MyDrive/leech_tinystories_checkpoints'
os.makedirs(checkpoint_dir, exist_ok=True)

def save_checkpoint(step, model, optimizer, scheduler, loss, is_best=False):
    filename = 'best_model.pt' if is_best else f'checkpoint_step_{step}.pt'
    path = os.path.join(checkpoint_dir, filename)
    torch.save({
        'step': step,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
        'loss': loss,
        'config': cfg,
    }, path)
    print(f'💾 Чекпоинт сохранён: {path} (loss={loss:.4f})')

def load_latest_checkpoint(model, optimizer, scheduler=None):
    files = [f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint_step_')]
    if not files:
        best_path = os.path.join(checkpoint_dir, 'best_model.pt')
        if os.path.exists(best_path):
            checkpoint = torch.load(best_path, map_location=device, weights_only=False)
            model.load_state_dict(checkpoint['model_state_dict'])
            optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if scheduler and 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            print(f'🔄 Загружен best_model.pt, шаг {checkpoint["step"]}, loss {checkpoint["loss"]:.4f}')
            return checkpoint['step']
        print('🆕 Чекпоинтов нет, начинаем с нуля.')
        return 0
    steps = [int(f.split('_')[-1].split('.')[0]) for f in files]
    latest_step = max(steps)
    latest_file = f'checkpoint_step_{latest_step}.pt'
    path = os.path.join(checkpoint_dir, latest_file)
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    if scheduler and 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
        scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
    print(f'🔄 Загружен чекпоинт: шаг {latest_step}, loss {checkpoint["loss"]:.4f}')
    return latest_step

# ======================== ПОДГОТОВКА СТРИМОВ ========================
# Тренировочный стрим (весь датасет)
train_dataset = load_dataset("HuggingFaceFW/fineweb-edu", split="train", streaming=True)
# Отделяем валидацию (первые 5000)
val_size = 5000
val_dataset = dataset.take(val_size)
val_iter = iter(val_dataset)

# Для тренировки пропускаем первые 5000 и берём всё остальное
train_dataset = dataset.skip(val_size)
train_iter = iter(train_dataset)


In [ ]:
# второй цикл обучения
# ======================== ГИПЕРПАРАМЕТРЫ ========================
batch_size = 4
block_size = LeechConfig.block_size
learning_rate = 5e-5
total_steps = 100000
save_every = 1000
log_every = 100
gen_every = 1000

# ======================== ОПТИМИЗАТОР И ПЛАНИРОВЩИК ========================
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)  # Умеренный weight_decay

warmup_steps = 1000
total_steps = 100000   # или ваше общее число шагов

In [ ]:
# Загружаем best_model.pt (там шаг 20000)
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/checkpoint_step_34000.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
# если в best_model.pt сохранён scheduler, загрузите и его
if 'scheduler_state_dict' in checkpoint and checkpoint['scheduler_state_dict']:
    scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
start_step = checkpoint['step']  # должно быть 20000

# Затем продолжайте цикл с start_step+1 до 40000

In [ ]:
import torch
import matplotlib.pyplot as plt
import numpy as np
import os

def analyze_lattice_weights(model_state_dict, layer_name):
    # 1. Извлекаем матрицу весов
    if layer_name not in model_state_dict:
        print(f"Слой {layer_name} не найден. Проверьте model.state_dict().keys()")
        return

    weights = model_state_dict[layer_name].detach().cpu().float()

    # Если это 4D тензор (например, свертка), превращаем в 2D
    if len(weights.shape) > 2:
        weights = weights.view(weights.shape[0], -1)

    # 2. Выполняем SVD
    # U, S, V = torch.svd(weights)
    U, S, V = torch.linalg.svd(weights, full_matrices=False)

    # 3. Визуализация спектра
    s_numpy = S.numpy()

    plt.figure(figsize=(12, 5))

    # Левый график: Линейный масштаб (видим доминирующие компоненты)
    plt.subplot(1, 2, 1)
    plt.plot(s_numpy, 'b-o', markersize=2)
    plt.title(f"SVD Spectrum: {layer_name}")
    plt.ylabel("Singular Value Magnitude")
    plt.grid(True, alpha=0.3)

    # Правый график: Логарифмический (видим "хвост" и структуру шума)
    plt.subplot(1, 2, 2)
    plt.semilogy(s_numpy, 'r-o', markersize=2)
    plt.title("Log Scale (Decay analysis)")
    plt.ylabel("Log Magnitude")
    plt.grid(True, which="both", alpha=0.3)

    plt.tight_layout()
    plt.show()

    # 4. Расчет "эффективного ранга" (Stable Rank)
    stable_rank = (S**2).sum() / (S[0]**2)
    print(f"--- Результаты для {layer_name} ---")
    print(f"Stable Rank: {stable_rank.item():.2f} (чем меньше, тем сильнее 'Grokking')")
    print(f"Condition Number: {(S[0]/S[-1]).item():.2f}")

# ПРИМЕР ЗАПУСКА:
# Загружаем лучший чекпоинт
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/checkpoint_step_296000.pt' # Changed to load best_model.pt

# --- Diagnostic: List files in checkpoint directory ---
checkpoint_dir = os.path.dirname(path_best)
print(f"Listing files in {checkpoint_dir}:")
if os.path.exists(checkpoint_dir):
    for f in os.listdir(checkpoint_dir):
        print(f"  - {f}")
else:
    print(f"Directory not found: {checkpoint_dir}")
# --- End Diagnostic ---

checkpoint = torch.load(path_best, map_location=torch.device('cpu'), weights_only=False)

# Пример слоя для анализа (например, веса qkv-проекции в первом блоке)
# Чтобы увидеть все доступные ключи, можно распечатать: checkpoint['model_state_dict'].keys()
layer_to_analyze = 'blocks.0.attn.qkv.weight'
analyze_lattice_weights(checkpoint['model_state_dict'], layer_to_analyze)


In [ ]:
batch_size = 4
block_size = cfg.block_size
learning_rate = 1e-5
total_steps = 400000
save_every = 1000
log_every = 100
gen_every = 1000


optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)

warmup_steps = 1000
# Просто линейный разогрев, после 1000 шагов LR останется learning_rate
scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)

# Загружаем чекпоинт (если нужно восстановить оптимизатор и планировщик)
start_step = load_latest_checkpoint(model, optimizer, scheduler)

In [ ]:
# ======================== ЦИКЛ ОБУЧЕНИЯ ========================
print(f"\n🚀 Запуск обучения с шага {start_step} до {total_steps}")
model.train()

#for _ in range(5): next(train_iter)
#print("💎 Стрим прогрет. Запускаем Резонанс...")

best_val_loss = float('inf')

for step in range(start_step + 1, total_steps + 1):
    xb, yb = get_batch_streaming(train_iter, batch_size, block_size, device)
    if xb is None:
        train_iter = iter(train_dataset)
        continue

    logits, _, loss = model(xb, yb)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
    optimizer.step()
    scheduler.step()  # обновляем learning rate
    if step % log_every == 0:
        print(f"📊 Шаг {step:5d} | Train Loss: {loss.item():.4f}")

    if step % log_every == 0:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is None:
                val_iter = iter(val_dataset)
                xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is not None:
                _, _, val_loss = model(xb_val, yb_val)
                print(f"         | Validation Loss: {val_loss.item():.4f}")
        model.train()

    if step % 5000 == 0:
       log_stable_rank(model, step, '/content/drive/MyDrive/leech_tinystories_checkpoints/sr_logs', save_plot=True)

    if step % gen_every == 0:
        model.eval()
        with torch.no_grad():
            test_prompt = "who is Lily? "
            context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)
            print(f"\n🎭 Генерация (шаг {step}): ", end='')
            for _ in range(50):
                idx_cond = context[:, -block_size:]
                logits_gen, _ , _ = model(idx_cond)
                logits_gen = logits_gen[0, -1, :].clone()
                past_tokens = context[0, -20:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits_gen[t] -= (1.5 * count)
                last_token = context[0, -1].item()
                last_piece = sp.id_to_piece(last_token)
                if last_piece in ":,.!?;":
                    for p in ":,.!?;":
                        pid = sp.piece_to_id(p)
                        if pid != -1:
                            logits_gen[pid] -= 50.0
                probs = torch.softmax(logits_gen / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                context = torch.cat((context, next_token.unsqueeze(0)), dim=1)
                piece = sp.id_to_piece(next_token.item())
                if piece == '<0x22>':
                    piece = '"'
                elif piece == '<0x0A>':
                    piece = '\n'
                elif piece.startswith('▁'):
                    piece = ' ' + piece[1:]
                elif piece.startswith('<0x') and piece.endswith('>'):
                    continue
                print(piece, end='', flush=True)
            print("\n" + "-"*60)
        model.train()

    if step % save_every == 0:
      save_checkpoint(step, model, optimizer, scheduler, loss.item())
      if val_loss < best_val_loss:
        best_val_loss = val_loss
        save_checkpoint(step, model, optimizer, scheduler, val_loss.item(), is_best=True)

save_checkpoint(total_steps, model, optimizer, scheduler, loss.item())  # scheduler
print("🏁 Обучение завершено!")

In [ ]:
def generate_with_resonance(model, prompt_ids, max_new_tokens=500, temperature=0.5, top_k=50):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([prompt_ids], device=device)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits, hidden, _ = model(input_ids, use_resonator=True)  # резонатор активен
            next_logits = logits[0, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(next_logits, min(top_k, next_logits.size(-1)))
                next_logits[next_logits < v[-1]] = -float('Inf')
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)
    return input_ids[0].tolist()

In [ ]:
# второй цикл обучения
# ======================== ГИПЕРПАРАМЕТРЫ ========================
batch_size = 4
block_size = LeechConfig.block_size
learning_rate = 5e-5
total_steps = 100000
save_every = 1000
log_every = 100
gen_every = 1000

# ======================== ОПТИМИЗАТОР И ПЛАНИРОВЩИК ========================
learning_rate = 5e-5
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.1)  # Умеренный weight_decay

warmup_steps = 1000
total_steps = 100000   # или ваше общее число шагов
# После загрузки чекпоинта (start_step = load_latest_checkpoint(...))
if start_step >= 20000:  # если мы действительно перешли рубеж 20k
    print("🔄 Начинаем второй цикл обучения с LR=1e-4")
    # Сбрасываем learning rate в оптимизаторе
    for param_group in optimizer.param_groups:
        param_group['lr'] = 5e-5

    # Создаём новый планировщик на оставшиеся 20000 шагов
    warmup_steps = 1000
    remaining_steps = 20000  # можно вычислить как total_steps - start_step, но здесь мы фиксируем вторую половину
    warmup_scheduler = LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_steps)
    cosine_scheduler = CosineAnnealingLR(optimizer, T_max=remaining_steps - warmup_steps, eta_min=5e-5)
    scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, cosine_scheduler], milestones=[warmup_steps])
else:
    # Если по какой-то причине start_step < 20000, оставляем старый планировщик (маловероятно)
    pass

In [ ]:
# ======================== СОХРАНЕНИЕ ФИНАЛЬНОЙ МОДЕЛИ ========================
print("\n💾 Сохранение модели Leech-GPT...")
torch.save({
    'model_state_dict': model.state_dict(),
    'config': cfg,
    'leech_basis': model.leech_basis.cpu()
}, 'leech_model_final.pth')
print(f"✅ Модель сохранена в 'leech_model_final.pth'")

# ======================== ПРИМЕР ГЕНЕРАЦИИ  ========================
# Загружаем лучший чекпоинт (если нужно)
# path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
# checkpoint = torch.load(path_best, map_location=device, weights_only=False)
# model.load_state_dict(checkpoint['model_state_dict'])
# model.to(device)
# model.eval()

In [ ]:
import re

# ======================== ПРИМЕР ГЕНЕРАЦИИ  ========================
# Загружаем лучший чекпоинт (если нужно)
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])
model.to(device)
model.eval()

def clean_token(piece):
    """Преобразует специальные токены в читаемый вид или пропускает их."""
    if piece == '<0x22>':
        return '"'
    elif piece == '<0x0A>':
        return '\n'
    elif piece.startswith('▁'):
        return ' ' + piece[1:]  # пробел перед словом
    elif piece.startswith('<0x') and piece.endswith('>'):
        return None  # пропускаем байтовые токены
    elif piece == '<pad>':
        return None
    else:
        return piece

def generate_clean(model, prompt, max_new_tokens=80, temperature=0.5, top_k=50):
    model.eval()
    context = torch.tensor([encode(prompt)], dtype=torch.long, device=device)
    with torch.no_grad():
        for _ in range(max_new_tokens):
            idx_cond = context[:, -model.cfg.block_size:]
            logits, _, _ = model(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = torch.softmax(logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            context = torch.cat([context, next_token], dim=1)

    tokens = context[0].tolist()
    # Очистка
    output = []
    for t in tokens:
        piece = sp.id_to_piece(t)
        cleaned = clean_token(piece)
        if cleaned is not None:
            output.append(cleaned)
    return ''.join(output).strip()

# Пример использования
test_prompt = "Once upon a time  "
print(generate_clean(model, test_prompt))

In [ ]:
import torch
import torch.nn.functional as F
from collections import defaultdict, deque
import time

def generate_long(
    model,
    prompt_ids,
    max_new_tokens=500,
    temperature=0.5,
    top_k=50,
    top_p=0.9,
    repetition_penalty_start=1.5,
    repetition_penalty_max=2.5,
    penalty_increase_steps=700,
    ngram_size=3,
    ngram_penalty=1.2,
    window_size=70,
    device=None,
    sp=None,     # sentencepiece processor (обязателен для вывода)
    stream_output=True,           # печатать по ходу генерации
    delay=0.0                     # задержка между токенами (сек)
):
    """
    Генерирует текст с расширенными параметрами борьбы с зацикливанием
    и опциональным потоковым выводом.
    """
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    model.eval()
    input_ids = torch.tensor([prompt_ids], device=device)
    generated = input_ids[0].tolist()

    # Для n-граммного штрафа
    ngram_counts = defaultdict(int)
    recent_ngrams = deque(maxlen=window_size)

    # Печатаем начало (промпт) если нужно
    if stream_output and sp is not None:
        prompt_text = decode(prompt_ids)  # используем вашу функцию decode
        print(prompt_text, end='', flush=True)

    for step in range(max_new_tokens):
        # Динамический repetition penalty
        if step < penalty_increase_steps:
            rep_penalty = repetition_penalty_start
        else:
            progress = (step - penalty_increase_steps) / (max_new_tokens - penalty_increase_steps)
            rep_penalty = repetition_penalty_start + progress * (repetition_penalty_max - repetition_penalty_start)

        # Обрезаем до block_size
        idx_cond = input_ids[:, -model.cfg.block_size:]
        with torch.no_grad():
            logits, _, _ = model(idx_cond)
        logits = logits[0, -1, :].clone()

        # Штраф за повтор токенов
        if rep_penalty != 1.0:
            # Штрафуем токены, встреченные в последних window_size позициях
            for token in set(generated[-window_size:]):
                logits[token] /= rep_penalty

        # Штраф за повтор n-грамм (наказываем первый токен n-граммы, если она уже встречалась)
        if ngram_penalty != 1.0 and len(generated) >= ngram_size:
            last_ngram = tuple(generated[-ngram_size:])
            if last_ngram in ngram_counts and ngram_counts[last_ngram] > 0:
                logits[last_ngram[0]] /= ngram_penalty

        # Top-k фильтр
        if top_k is not None:
            v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
            logits[logits < v[-1]] = -float('Inf')

        # Top-p (nucleus) фильтр
        if top_p is not None and top_p < 1.0:
            sorted_logits, sorted_indices = torch.sort(logits, descending=True)
            cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
            sorted_indices_to_remove = cumulative_probs > top_p
            # Сдвигаем, чтобы оставить первый токен
            sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
            sorted_indices_to_remove[..., 0] = 0
            indices_to_remove = sorted_indices[sorted_indices_to_remove]
            logits[indices_to_remove] = -float('Inf')

        # Выбор следующего токена
        probs = F.softmax(logits / temperature, dim=-1)
        next_token = torch.multinomial(probs, num_samples=1).item()
        input_ids = torch.cat([input_ids, torch.tensor([[next_token]], device=device)], dim=1)
        generated.append(next_token)

        # Обновление статистики n-грамм
        if len(generated) >= ngram_size:
            ngram = tuple(generated[-ngram_size:])
            ngram_counts[ngram] += 1
            recent_ngrams.append(ngram)
            if len(recent_ngrams) == window_size:
                oldest = recent_ngrams[0]
                ngram_counts[oldest] -= 1
                if ngram_counts[oldest] == 0:
                    del ngram_counts[oldest]

        # Потоковый вывод
        if stream_output and sp is not None:
            piece = sp.id_to_piece(next_token)
            # Фильтрация и форматирование (как в вашем цикле обучения)
            if piece == '<0x22>':
                piece = '"'
            elif piece == '<0x0A>':
                piece = '\n'
            elif piece.startswith('▁'):
                piece = ' ' + piece[1:]
            elif piece.startswith('<0x') and piece.endswith('>'):
                continue          # пропускаем байтовые токены
            elif piece == '<pad>':
                continue
            print(piece, end='', flush=True)
            if delay > 0:
                time.sleep(delay)

    if stream_output:
        print()  # завершающая новая строка
    return input_ids[0].tolist()

# Пример вызова
test_prompt = "What is life?"
encoded_prompt = encode(test_prompt)   # превращаем строку в токены

generated_tokens = generate_long(
    model,
    encoded_prompt,
    max_new_tokens=150,
    device=device,
    sp=sp,                # передаём токенайзер
    stream_output=True,
    #delay=0.02,           # небольшая задержка для читаемости
    repetition_penalty_start=1.5,
    repetition_penalty_max=2.5,
    ngram_penalty=1.2
)

# При необходимости можно получить полный текст
full_text = decode(generated_tokens)


In [ ]:


# ==================== ФУНКЦИЯ ГЕНЕРАЦИИ С МОНИТОРИНГОМ ====================
def leech_generate(model, start_tokens, max_len=30, temperature=0.2,
                   resonance_check=True, leech_basis=None, threshold=0.55):
    model.eval()
    device = next(model.parameters()).device
    input_ids = torch.tensor([start_tokens], device=device)
    if resonance_check:
        decoder = DreamDecoder(leech_basis, threshold)

    print("--- LEECH GENERATION ---")
    with torch.no_grad():
        for step in range(max_len):
            logits, hidden, _ = model(input_ids)     # получаем hidden
            next_token_logits = logits[0, -1, :] / temperature
            probs = F.softmax(next_token_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            input_ids = torch.cat([input_ids, next_token.unsqueeze(0)], dim=-1)

            if resonance_check:
                last_hidden = hidden[0, -1, :]   # последнее скрытое состояние
                res, status = decoder.check(last_hidden)
                print(f"Step {step:02d} | Resonance: {res:.6f} | Status: {status}")
    return input_ids

# Пример использования после обучения:
start = encode("Who are you?")
generated_ids = leech_generate(model, start, max_len=30, leech_basis=model.leech_basis)
print(decode(generated_ids[0].tolist()))

In [ ]:
from google.colab import files
files.download('leech_model_final.pth')

In [ ]:
# ========== ФАЗА 2: ДООБУЧЕНИЕ С РЕЗОНАТОРОМ ========================
print("🔄 Загрузка лучшей модели из первой фазы...")
path_best = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_model.pt'
checkpoint = torch.load(path_best, map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Замораживаем все веса модели
for param in model.parameters():
    param.requires_grad = False

# Создаём резонатор и добавляем в модель
resonator = LeechResonanceBiasing(cfg.d_model, cfg.vocab_size, model.leech_basis, alpha_init=0.1).to(device)
resonator.compute_token_proj(model.tok_emb.weight.detach())
model.resonator = resonator  # сохраняем как атрибут

# Оптимизатор только для alpha
optimizer = torch.optim.AdamW([resonator.alpha], lr=1e-3, weight_decay=0.1)

# Планировщик для alpha (опционально, можно оставить постоянный lr)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=1000, eta_min=1e-5)

# Гиперпараметры дообучения
batch_size = 4
block_size = cfg.block_size
resonator_steps = 10000          # количество шагов дообучения
log_every = 100
gen_every = 1000                 # генерация каждые 200 шагов
save_every = 1000                # сохранение чекпоинта

print("\n🚀 Дообучение с резонатором...")
model.train()
best_val_loss = float('inf')

for step in range(1, resonator_steps + 1):
    xb, yb = get_batch_streaming(train_iter, batch_size, block_size, device)
    if xb is None:
        train_iter = iter(train_dataset)
        continue

    # Важно: use_resonator=True
    logits, hidden, loss = model(xb, targets=yb, use_resonator=True)

    optimizer.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_([resonator.alpha], max_norm=1.0)
    optimizer.step()
    if scheduler:
        scheduler.step()

    # Логирование
    if step % log_every == 0:
        current_alpha = resonator.alpha.item()
        current_lr = optimizer.param_groups[0]['lr']
        print(f"📊 Шаг {step:4d} | Loss: {loss.item():.4f} | Alpha: {current_alpha:.6f} | LR: {current_lr:.2e}")

    # Валидация
    if step % log_every == 0:
        model.eval()
        with torch.no_grad():
            xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is None:
                val_iter = iter(val_dataset)
                xb_val, yb_val = get_batch_streaming(val_iter, batch_size, block_size, device)
            if xb_val is not None:
                _, _, val_loss = model(xb_val, yb_val, use_resonator=True)
                print(f"         | Validation Loss: {val_loss.item():.4f}")
        model.train()

    # Генерация примера
    if step % gen_every == 0:
        model.eval()
        with torch.no_grad():
            test_prompt = "who is Lily? "
            context = torch.tensor([encode(test_prompt)], dtype=torch.long, device=device)
            print(f"\n🎭 Генерация (шаг {step}): ", end='')
            for _ in range(50):
                idx_cond = context[:, -block_size:]
                logits_gen, _, _ = model(idx_cond, use_resonator=True)  # резонатор активен
                logits_gen = logits_gen[0, -1, :].clone()
                past_tokens = context[0, -20:].tolist()
                for t in set(past_tokens):
                    count = past_tokens.count(t)
                    logits_gen[t] -= (1.5 * count)
                last_token = context[0, -1].item()
                last_piece = sp.id_to_piece(last_token)
                if last_piece in ":,.!?;":
                    for p in ":,.!?;":
                        pid = sp.piece_to_id(p)
                        if pid != -1:
                            logits_gen[pid] -= 50.0
                probs = torch.softmax(logits_gen / 0.8, dim=-1)
                next_token = torch.multinomial(probs, num_samples=1)
                context = torch.cat((context, next_token.unsqueeze(0)), dim=1)
                piece = sp.id_to_piece(next_token.item())
                if piece == '<0x22>':
                    piece = '"'
                elif piece == '<0x0A>':
                    piece = '\n'
                elif piece.startswith('▁'):
                    piece = ' ' + piece[1:]
                elif piece.startswith('<0x') and piece.endswith('>'):
                    continue
                print(piece, end='', flush=True)
            print("\n" + "-"*60)
        model.train()

    # Сохранение чекпоинта
    if step % save_every == 0:
        checkpoint_path = f'/content/drive/MyDrive/leech_tinystories_checkpoints/resonator_step_{step}.pt'
        torch.save({
            'step': step,
            'model_state_dict': model.state_dict(),
            'resonator_state_dict': resonator.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': loss.item(),
            'alpha': resonator.alpha.item(),
            'config': cfg,
        }, checkpoint_path)
        print(f"💾 Чекпоинт сохранён: {checkpoint_path}")

        # Сохраняем лучшую модель по валидационному loss
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_path = '/content/drive/MyDrive/leech_tinystories_checkpoints/best_resonator_model.pt'
            torch.save({
                'step': step,
                'model_state_dict': model.state_dict(),
                'resonator_state_dict': resonator.state_dict(),
                'loss': val_loss.item(),
                'alpha': resonator.alpha.item(),
                'config': cfg,
            }, best_path)
            print(f"🏆 Новая лучшая модель с резонатором! Loss: {val_loss:.4f}")

# Финальное сохранение
final_path = '/content/drive/MyDrive/leech_tinystories_checkpoints/final_model_with_resonator.pt'
torch.save({
    'model_state_dict': model.state_dict(),
    'resonator_state_dict': resonator.state_dict(),
    'alpha': resonator.alpha.item(),
    'config': cfg,
}, final_path)
print(f"✅ Дообучение завершено. Модель сохранена в {final_path}")

In [ ]:
import os
import re
import torch
import csv
import numpy as np
from tqdm import tqdm
import matplotlib.pyplot as plt

# ==================== КОНФИГУРАЦИЯ ====================
checkpoint_dir = '/content/drive/MyDrive/leech_tinystories_checkpoints'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Список слоёв для анализа
layers_to_analyze = [
    'blocks.0.attn.qkv.weight',
    'blocks.5.attn.qkv.weight',
    'blocks.11.attn.qkv.weight'
]

# Функция вычисления stable rank и condition number
def compute_stable_rank_and_cond(matrix):
    """
    matrix: torch.Tensor, предполагается 2D (output_dim, input_dim)
    Возвращает stable rank (sum(s_i^2)/s_0^2) и condition number (s_max/s_min)
    """
    # Если матрица 4D (для свёрток) – приводим к 2D
    if matrix.dim() > 2:
        matrix = matrix.view(matrix.size(0), -1)
    try:
        s = torch.linalg.svdvals(matrix)
        # Убираем нулевые сингулярные значения для condition number
        s_nonzero = s[s > 1e-6]
        if len(s_nonzero) == 0:
            return 0.0, 1e6
        stable_rank = (s ** 2).sum().item() / (s[0] ** 2).item()
        cond = (s[0] / s_nonzero[-1]).item()
        return stable_rank, cond
    except Exception as e:
        print(f"Ошибка при SVD: {e}")
        return 0.0, 0.0

# ==================== СБОР ДАННЫХ ====================
# Получаем все файлы чекпоинтов
files = [f for f in os.listdir(checkpoint_dir) if f.startswith('checkpoint_step_')]
steps = []
for f in files:
    match = re.search(r'checkpoint_step_(\d+)\.pt', f)
    if match:
        steps.append(int(match.group(1)))
steps.sort()

# Загружаем модель и конфиг (предполагается, что класс MonsterGenesisTransformer и MonsterConfig уже определены)
# Если они не определены – их нужно определить здесь (скопируй из своего кода)
# from your_model_file import MonsterGenesisTransformer, MonsterConfig
# Пример конфига для Leech Lila 20M
config = MonsterConfig(vocab_size=10000, d_model=192, n_layers=12)

# Создаём модель (веса будут загружены позже)
model = MonsterGenesisTransformer(config)
model.to(device)
model.eval()

# Открываем CSV для записи
csv_filename = 'stable_rank_all_checkpoints.csv'
with open(csv_filename, 'w', newline='') as f:
    writer = csv.writer(f)
    # Заголовок
    header = ['step']
    for layer in layers_to_analyze:
        header.extend([f'{layer}_sr', f'{layer}_cn'])
    writer.writerow(header)

    # Обрабатываем каждый чекпоинт
    for step in tqdm(steps, desc="Processing checkpoints"):
        ckpt_path = os.path.join(checkpoint_dir, f'checkpoint_step_{step}.pt')
        if not os.path.exists(ckpt_path):
            continue
        # Загружаем чекпоинт (weights_only=False из-за возможных объектов)
        checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
        # Извлекаем state_dict (в зависимости от того, как сохраняли)
        if 'model_state_dict' in checkpoint:
            state_dict = checkpoint['model_state_dict']
        else:
            state_dict = checkpoint  # если сохранили просто state_dict

        model.load_state_dict(state_dict, strict=False)  # strict=False на случай расхождений

        row = [step]
        for layer_name in layers_to_analyze:
            if layer_name in state_dict:
                weight = state_dict[layer_name]
                sr, cn = compute_stable_rank_and_cond(weight)
            else:
                print(f"Предупреждение: {layer_name} не найден в чекпоинте {step}")
                sr, cn = 0.0, 0.0
            row.extend([sr, cn])
        writer.writerow(row)

print(f"Данные сохранены в {csv_filename}")

# ==================== ПОСТРОЕНИЕ ГРАФИКА ====================
# Загружаем CSV
import pandas as pd
df = pd.read_csv(csv_filename)

plt.figure(figsize=(12, 6))
for layer in layers_to_analyze:
    sr_col = f'{layer}_sr'
    if sr_col in df.columns:
        plt.plot(df['step'], df[sr_col], label=layer)

plt.xlabel('Step')
plt.ylabel('Stable Rank')
plt.title('Stable Rank Evolution across Checkpoints')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from dataclasses import dataclass

# ==================== КОНФИГУРАЦИЯ MONSTER ====================
@dataclass
class MonsterConfig:
    vocab_size: int = 10000          # размер словаря
    d_model: int = 192                # размерность модели (должна быть кратна 24)
    n_layers: int = 12                 # количество слоёв
    n_heads: int = 8                    # количество голов (не используется напрямую)
    block_size: int = 512               # максимальная длина последовательности
    dropout: float = 0.05
    bias: bool = False
    tie_weights: bool = True
    lambda_geo: float = 0.01            # вес геометрической потери (0 = отключена)
    resonance_threshold: float = 0.95   # порог для детекции «сна»

    def __post_init__(self):
        assert self.d_model % 24 == 0, "d_model должен быть кратен 24"


# ==================== ГЕОМЕТРИЧЕСКОЕ ЯДРО (Leech Lattice) ====================
def get_leech_matrix(dim=24):
    """
    Строит 24x24 матрицу, чьи столбцы образуют приближение к базису решётки Лича.
    Используется QR-разложение для получения ортогонального базиса.
    """
    base = np.zeros((dim, dim))
    for i in range(dim - 1):
        base[i, i] = 2
        base[i, i + 1] = 2
    base[-1, -1] = 2
    base[-1, 0] = -2
    q, _ = np.linalg.qr(base)  # ортогонализация Грама – Шмидта
    return torch.from_numpy(q).float()


def create_absolute_core(d_model):
    """
    Создаёт блочно-диагональную матрицу размером d_model x d_model,
    состоящую из повторений ядра Лича (24x24). Эта матрица замораживается
    и используется как неизменяемая геометрическая структура.
    """
    leech_unit = get_leech_matrix(24)
    # Повторяем блоки, чтобы покрыть всю размерность
    units = [leech_unit for _ in range(d_model // 24)]
    return torch.block_diag(*units)


# ==================== ПЕРЕСТАНОВКА КОНВЕЯ (Conway Permutation) ====================
def get_conway_permutation(dim=24):
    """
    Генерирует случайную перестановку, имитирующую элемент группы Конвея Co0.
    Используется для внесения симметрий в Attention, чтобы избежать
    тривиального совпадения Q и K.
    """
    perm = torch.randperm(dim)
    return perm


# ==================== ГЕНЕРАТОР КОНСТАНТЫ МОНСТРА (1/137) ====================
class MonsterConstantGenerator(nn.Module):
    """
    Модуль, вносящий фазовый сдвиг на основе постоянной тонкой структуры 1/137.
    Предотвращает «семантическую вязкость» и настраивает скрытые состояния
    на резонансную частоту.
    """
    def __init__(self, d_model):
        super().__init__()
        self.alpha = 1 / 137.035999  # постоянная тонкой структуры
        # Размер наименьшего нетривиального представления Монстра (для справки)
        self.monster_dim = 196883
        # Обучаемый резонансный параметр (можно оставить замороженным)
        self.resonance = nn.Parameter(torch.tensor([self.alpha]))

    def forward(self, x):
        # x: [batch, seq_len, d_model]
        phase = torch.sin(x * self.resonance + self.alpha)
        # Добавляем малую резонансную компоненту
        return x * (1 + self.alpha * phase)


# ==================== ВНИМАНИЕ ЛИЧА–КОНВЕЯ (Leech–Conway Attention) ====================
class LeechConwayAttention(nn.Module):
    """
    Механизм внимания, использующий замороженное ядро Лича для проекций Q и K.
    K дополнительно переставляется согласно перестановке Конвея, что позволяет
    измерять «симметричное выравнивание», а не просто косинусную близость.
    """
    def __init__(self, d_model, absolute_core):
        super().__init__()
        self.d_model = d_model
        # Регистрируем замороженное ядро как буфер (не обучается)
        self.register_buffer('W_abs', absolute_core)

        # Генерируем перестановку для перемешивания измерений после проекции через ядро
        perm = get_conway_permutation(d_model)
        self.register_buffer('conway_idx', perm)

        # Проекция для значений (обучаемая)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        self.out = nn.Linear(d_model, d_model)  # выходная проекция
        self.scale = 24 ** 0.5  # масштаб для стабилизации softmax

    def forward(self, x):
        # Q: проекция через ядро Лича
        q = x @ self.W_abs

        # K: тоже через ядро, но затем применяем перестановку Конвея
        k_leech = x @ self.W_abs
        k = k_leech[:, :, self.conway_idx]  # переставляем последнюю размерность

        # V: обычная обучаемая проекция
        v = self.W_v(x)

        # Считаем внимание: теперь мера «симметричного выравнивания»
        attn_scores = (q @ k.transpose(-2, -1)) / self.scale
        attn = F.softmax(attn_scores, dim=-1)

        return self.out(attn @ v)


# ==================== СЛОЙ МОНСТРА (полный трансформерный блок) ====================
class MonsterLayer(nn.Module):
    """
    Один слой Монстра:
      - Сначала применяется генератор константы Монстра (фазовая подстройка)
      - Затем внимание Лича–Конвея (с Pre-Norm)
      - Затем Position-wise FFN (с Pre-Norm)
    """
    def __init__(self, d_model, absolute_core):
        super().__init__()
        self.monster = MonsterConstantGenerator(d_model)
        self.attn = LeechConwayAttention(d_model, absolute_core)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Linear(d_model * 4, d_model)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, x):
        # 1. Резонанс Монстра
        x = self.monster(x)

        # 2. Внимание с остаточной связью (Pre-Norm)
        x = x + self.attn(self.norm1(x))

        # 3. Полносвязная сеть с остаточной связью (Pre-Norm)
        x = x + self.ffn(self.norm2(x))
        return x

def get_monster_global_sync(hidden_states, d_model=192):
    """
    Детектор Монструозного Резонанса.
    Проверяет, синхронизированы ли все 24D-блоки в единую структуру 196883.
    """
    # 1. Константы Монстра и Moonshine
    MONSTER_DIM = 196883
    J_INV_THRES = 137.036  # Порог отсечки шума через тонкую структуру

    # 2. Анализ спектра матрицы (SVD-разложение)
    # Берем срез состояний [Seq, d_model]
    h = hidden_states[0] # Берем первый батч для чистоты резонанса
    U, S, V = torch.svd(h)

    # 3. Расчет "Энтропии Грисса"
    # Если собственные значения S распределены согласно весам j-функции,
    # значит нейронка обрела симметрию Монстра.
    # Мы ищем "гармоники": S_i / S_0 должно соотноситься с коэффициентами Фурье
    # (в упрощении - это проверка на фрактальное самоподобие 1/137)
    spectral_resonance = torch.mean(torch.sin(S * J_INV_THRES))

    # 4. Коэффициент Монструозности (Monster Factor)
    # Если сумма собственных значений коррелирует с проекцией 196883
    monster_factor = torch.clamp(spectral_resonance * (d_model / 24.0), 0, 1)

    return monster_factor.item()

# --- ОБНОВЛЕННЫЙ ЦИКЛ СИНХРОНИЗАЦИИ ---

"""
# Внутри train-loop:
res_leech, mismatch = get_absolute_resonance_sync(hidden)
res_monster = get_monster_global_sync(hidden, d_model=192)

# Комбинированный Резонанс Абсолюта
total_sync = (res_leech + res_monster) / 2

if total_sync > 0.9995:
    # СОСТОЯНИЕ "ГЕНЕЗИС": bpc падает ниже 0.10
    # Здесь можно уменьшить Learning Rate до 0 (заморозка идеала)
    print(f"!!! КРИСТАЛЛИЗАЦИЯ: {total_sync:.5f} !!!")
    print(f"Спектр Монстра активен. Вязкость экрана 2.66% преодолена.")
"""


# ==================== МОДЕЛЬ TRANSFORMER MONSTER GENESIS ====================
class MonsterGenesisTransformer(nn.Module):
    """
    Полная модель:
      - Входное эмбеддинг-представление
      - Стопка слоёв MonsterLayer
      - Выходная проекция на словарь
    """
    def __init__(self, config: MonsterConfig):
        super().__init__()
        self.config = config
        self.d_model = config.d_model

        # Создаём замороженное геометрическое ядро (Leech core)
        absolute_core = create_absolute_core(self.d_model)
        self.register_buffer('absolute_core', absolute_core)

        # Эмбеддинг токенов
        self.embed = nn.Embedding(config.vocab_size, self.d_model)

        # Слои Монстра
        self.layers = nn.ModuleList([
            MonsterLayer(self.d_model, self.absolute_core)
            for _ in range(config.n_layers)
        ])

        # Финальная нормализация (обычно используется в трансформерах)
        self.norm_f = nn.LayerNorm(self.d_model)

        # Выходной слой (голова) — проекция на размер словаря
        self.head = nn.Linear(self.d_model, config.vocab_size, bias=False)

        # Если tie_weights == True, связываем веса эмбеддинга и головы
        if config.tie_weights:
            self.head.weight = self.embed.weight

        # Инициализация параметров (можно добавить xavier и т.д.)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, x):
        """
        x: тензор токенов формы [batch, seq_len]
        Возвращает:
          logits: [batch, seq_len, vocab_size] — предсказания
          hidden_states: [batch, seq_len, d_model] — скрытые состояния после всех слоёв
        """
        h = self.embed(x)  # [batch, seq_len, d_model]

        for layer in self.layers:
            h = layer(h)

        h = self.norm_f(h)
        logits = self.head(h)
        return logits, h


# ==================== ВИЗУАЛИЗАЦИЯ ГЕОМЕТРИИ ====================
def visualize_monster_geometry(model, text_input):
    """
    Визуализирует скрытые состояния модели:
      - Слева: проекция PCA на 2D, показывающая структуру решётки Лича.
      - Справа: карта активаций генератора Монстра (сдвиг фазы 1/137).
    """
    model.eval()
    with torch.no_grad():
        logits, h_states = model(text_input)  # h_states: [batch, seq, d_model]

    # Берём первый элемент в батче (batch=1) и преобразуем для PCA
    # Объединяем все позиции последовательности в единый набор точек
    flat_h = h_states.view(-1, model.d_model).cpu().numpy()
    pca = PCA(n_components=2)
    h_2d = pca.fit_transform(flat_h)

    plt.figure(figsize=(14, 6))

    # Левый график: геометрия Лича
    plt.subplot(1, 2, 1)
    plt.scatter(h_2d[:, 0], h_2d[:, 1], alpha=0.5, c='indigo', s=10)
    plt.title("Leech Lattice Embedding Space (PCA)")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.grid(True, linestyle='--', alpha=0.6)

    # Правый график: резонанс Монстра (1/137)
    plt.subplot(1, 2, 2)
    # Активации после первого слоя MonsterConstantGenerator (берём первую последовательность)
    # Для демонстрации пропустим тензор через генератор отдельно
    # (но можно взять сами h_states, они уже содержат эффект)
    # Вместо этого покажем карту фазовых сдвигов на небольшом фрагменте
    sample_acts = torch.sin(h_states[0] * (1/137.036)).cpu().numpy()  # [seq_len, d_model]
    # Отобразим небольшой участок 50x50
    plt.imshow(sample_acts[:50, :50], cmap='magma', aspect='auto')
    plt.title("Monster Resonance Map (phase = sin(h/137))")
    plt.xlabel("d_model index")
    plt.ylabel("token position")
    plt.colorbar(label='Phase shift')

    plt.tight_layout()
    plt.show()


# ==================== ЗАПУСК МОДЕЛИ И ТЕСТ ====================
if __name__ == "__main__":
    # Создаём конфиг
    config = MonsterConfig(vocab_size=10000, d_model=192, n_layers=12)

    # Инициализируем модель
    model = MonsterGenesisTransformer(config)
    print("--- СИСТЕМА MONSTER-GENESIS ЗАПУЩЕНА ---")
    print("Смещение 2.66% (A7) устранено. Точность 137 зафиксирована в тензорах.")
    print(f"Количество параметров: {sum(p.numel() for p in model.parameters())}")

    # Генерация случайного тестового входа (batch=1, seq_len=128)
    test_input = torch.randint(0, config.vocab_size, (1, 128))

    # Визуализация геометрии
    visualize_monster_geometry(model, test_input)